In [1]:
import scanpy as sc

In [2]:
import pandas as pd
import numpy as np

In [3]:
spectra = sc.read_h5ad('adata_spectra.h5ad')


In [4]:
spectra.obs.head()

,n_genes,sample,library,Processing,Treatment,10x kit,percent_mito,n_counts,scrublet_score,genotype,...,Stage,phase,dataset,Biopsy_type,Tissue_sampled,Age,Endometrial_pathology,cell_type,lineage,label_long
HCA_A_RepT_RNA13247830_AAACCTGAGCGATGAC-Mareckova,1981,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.036783,3956.0,0.033156,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,ePV_1a,Mesenchymal,19 | ePV 1a
HCA_A_RepT_RNA13247830_AAACCTGAGCTAGGCA-Mareckova,3492,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.042249,9546.0,0.029188,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,eStromal,Mesenchymal,23 | eStromal
HCA_A_RepT_RNA13247830_AAACCTGCACCAACCG-Mareckova,2967,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.092998,7188.0,0.075581,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,Venous,Endothelial,32 | Venous
HCA_A_RepT_RNA13247830_AAACCTGCACCTGGTG-Mareckova,2597,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.035448,6029.0,0.069252,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,mPV,Mesenchymal,18 | mPV
HCA_A_RepT_RNA13247830_AAACCTGCATAGAAAC-Mareckova,3319,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.036459,8571.0,0.054963,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,Venous,Endothelial,32 | Venous


In [5]:
index_labels = spectra.uns['SPECTRA_overlap'].index
gene_weights = pd.DataFrame(spectra.uns['SPECTRA_factors'],
                            index= index_labels,
                            columns=spectra.var[spectra.var['spectra_vocab']].index)

sdata = sc.AnnData(X = spectra.obsm['SPECTRA_cell_scores'], 
                  obs = spectra.obs,
                  var = pd.DataFrame({"index_label" : index_labels})
                  )

sdata.var = sdata.var.set_index('index_label')

/software/cellgen/team292/mm58/venvs/lightning2/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [6]:
list(sdata.var.index)

['0-X-global-X-Androgen',
 '1-X-global-X-EGFR',
 '2-X-global-X-Estrogen',
 '3-X-global-X-Hypoxia',
 '4-X-global-X-JAK-STAT',
 '5-X-global-X-MAPK',
 '6-X-global-X-NFkB',
 '7-X-global-X-p53',
 '8-X-global-X-PI3K',
 '9-X-global-X-TGFb',
 '10-X-global-X-TNFa',
 '11-X-global-X-VEGF',
 '12-X-global-X-WNT',
 '13-X-global-X-13',
 '14-X-Arterial-X-14',
 '15-X-Ciliated-X-15',
 '16-X-Cycling-X-16',
 '17-X-Fibroblast_basalis-X-17',
 '18-X-Glandular-X-18',
 '19-X-Glandular_secretory-X-19',
 '20-X-Glandular_secretory_FGF7-X-20',
 '21-X-HOXA13-X-21',
 '22-X-Immune_Lymphoid-X-22',
 '23-X-Immune_Myeloid-X-23',
 '24-X-KRT5-X-24',
 '25-X-Luminal-X-25',
 '26-X-Lymphatic-X-26',
 '27-X-MUC5B-X-27',
 '28-X-SOX9_basalis-X-28',
 '29-X-SOX9_functionalis_I-X-29',
 '30-X-SOX9_functionalis_II-X-30',
 '31-X-SOX9_luminal-X-31',
 '32-X-Venous-X-32',
 '33-X-dHormones-X-33',
 '34-X-dStromal_early-X-34',
 '35-X-dStromal_late-X-35',
 '36-X-dStromal_mid-X-36',
 '37-X-eHormones-X-37',
 '38-X-ePV_1a-X-38',
 '39-X-ePV_1b-X-3

In [7]:
sdata.write_h5ad('spectra_latent.h5ad')

In [8]:
sc.pp.neighbors(sdata, use_rep = 'X')
sc.tl.umap(sdata)

In [ ]:
meta = pd.read_csv('/lustre/scratch126/cellgen/team361/mm58/tripso_reproducibility/02_benchmarking/endometrium/celltypes_metadata.csv')

In [ ]:
meta2 = meta[['Cellstate', 'Symbol', 'color']]
meta2['label_long'] = meta['Symbol'].astype(str) + ' | ' + meta['Cellstate']
meta2['label_long'] = meta2['label_long'].str.replace('_', ' ')

In [ ]:
meta2 = meta2.drop_duplicates()

In [ ]:
meta2 = meta2[meta2['color'] != '#fff401']

In [ ]:
color_dict = {k : v for k, v in zip(meta2['label_long'], meta2['color'])}

In [ ]:
# sdata.obs = sdata.obs.join(meta2[['label_long', 'Cellstate']].set_index('Cellstate'), on = 'cell_type')

In [ ]:
sdata.obs['label_long'] = pd.Categorical(sdata.obs['label_long'], 
                                         categories=meta2['label_long'], 
                                         ordered=True)


In [ ]:
sc.pl.umap(sdata, color='label_long', palette=color_dict, frameon = False)
